In [1]:
import pandas as pd 
import numpy as np

In [2]:
import requests

# Конфигурация
API_KEY = "ACMA:xKKO5TfuvIYOZZDaDzit2m2kOFTgTjL23S6pDH5Q:2d21d480"
URL = f"https://api.partner.market.yandex.ru/campaigns"

headers = {
    "Api-Key": API_KEY
}

response = requests.get(URL, headers=headers)
print(response.content)

b'{"campaigns":[{"domain":"Arnella design","id":148709175,"clientId":311369611,"business":{"id":214995523,"name":"Arnella design"},"placementType":"FBY","apiAvailability":"AVAILABLE"}],"pager":{"total":1,"from":1,"to":1,"currentPage":1,"pagesCount":1,"pageSize":1}}'


In [4]:
URL = f"https://api.partner.market.yandex.ru/campaigns"

headers = {
    "Api-Key": API_KEY
}

response = requests.get(URL, headers=headers)
print(response.json())

{'campaigns': [{'domain': 'Arnella design', 'id': 148709175, 'clientId': 311369611, 'business': {'id': 214995523, 'name': 'Arnella design'}, 'placementType': 'FBY', 'apiAvailability': 'AVAILABLE'}], 'pager': {'total': 1, 'from': 1, 'to': 1, 'currentPage': 1, 'pagesCount': 1, 'pageSize': 1}}


In [13]:
import requests
import pandas as pd
from datetime import datetime

# Ваши данные
API_KEY = "ACMA:xKKO5TfuvIYOZZDaDzit2m2kOFTgTjL23S6pDH5Q:2d21d480"
CAMPAIGN_ID = '148709175'  # ID вашей кампании на Яндекс.Маркет

# Настройки запроса
BASE_URL = 'https://api.partner.market.yandex.ru/'
HEADERS = {
    'Api-Key': f"{API_KEY}",
    # 'Content-Type': 'application/json'
}

# Проверка доступа
def check_auth():
    url = f'{BASE_URL}campaigns/{CAMPAIGN_ID}/auth/check'
    headers = {
        'Api-Key': f"{API_KEY}",
        # 'Content-Type': 'application/json'
    }
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            print("Аутентификация успешна!")
            return True
        else:
            print(f"Ошибка аутентификации: {response.status_code}")
            print(response.text)
            return False
    except Exception as e:
        print(f"Ошибка при проверке аутентификации: {e}")
        return False

if check_auth():
    print("Можно продолжать работу с API")
else:
    print("Проверьте API-ключ и Campaign ID")

def get_offer_mappings():
    """Получение информации о соответствии ваших товаров и товаров на Маркете"""
    url = f'{BASE_URL}campaigns/{CAMPAIGN_ID}/offer-mapping-entries'
    
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при запросе offer-mappings: {e}")
        return None

def get_offers():
    """Получение информации о ваших товарах"""
    url = f'{BASE_URL}campaigns/{CAMPAIGN_ID}/offers'
    params = {
        'page': 1 
    }
    
    try:
        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при запросе offers: {e}")
        return None

def get_orders():
    """Получение информации о заказах"""
    url = f'{BASE_URL}campaigns/{CAMPAIGN_ID}/orders'
    
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при запросе orders: {e}")
        return None

def save_to_excel(data, filename_prefix):
    """Сохранение данных в Excel файл"""
    if not data:
        return

    # Создаем имя файла с текущей датой
    current_date = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    filename = f"{filename_prefix}_{current_date}.xlsx"
    
    # Преобразуем данные в DataFrame и сохраняем
    df = pd.DataFrame(data)
    df.to_excel(filename, index=False)
    print(f"Данные сохранены в файл: {filename}")

def main():
    # Получаем информацию о товарах
    offer_mappings = get_offer_mappings()
    if offer_mappings:
        print(f"Получено {len(offer_mappings.get('result', {}).get('offerMappingEntries', []))} записей о товарах")
        save_to_excel(offer_mappings.get('result', {}).get('offerMappingEntries', []), 'offer_mappings')
    
    # Получаем информацию о предложениях
    offers = get_offers()
    if offers:
        print(f"Получено {len(offers.get('result', {}).get('offers', []))} предложений")
        save_to_excel(offers.get('result', {}).get('offers', []), 'offers')
    
    # Получаем информацию о заказах (опционально)
    orders = get_orders()
    if orders:
        print(f"Получено {len(orders.get('result', {}).get('orders', []))} заказов")
        save_to_excel(orders.get('result', {}).get('orders', []), 'orders')

if __name__ == '__main__':
    main()

Ошибка аутентификации: 404
{"errors":[{"code":"NOT_FOUND","message":"Resource not found"}],"status":"ERROR"}
Проверьте API-ключ и Campaign ID
Ошибка при запросе offer-mappings: 403 Client Error: Forbidden for url: https://api.partner.market.yandex.ru/campaigns/148709175/offer-mapping-entries
Ошибка при запросе offers: 405 Client Error: Method Not Allowed for url: https://api.partner.market.yandex.ru/campaigns/148709175/offers?page=1
Получено 0 заказов
